In [ ]:
!pip install google-cloud-bigquery

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

In [ ]:
!gcloud config list

[component_manager]
disable_update_check = True
[core]
account = zehraoguzturk1@gmail.com

Your active configuration is: [default]


In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="lastproject-465319")

In [ ]:
datasets = list(client.list_datasets())
for d in datasets:
    print(d.dataset_id)

In [ ]:
tables = list(client.list_tables("lastproject-465319.TEZ"))
for t in tables:
    print(t.table_id)

order_history
rfm_results


In [ ]:
query = """
SELECT
  CustomerID as customer_id,
  PurchaseDate as purchase_date,
  PurchaseAmount as total_price
FROM `lastproject-465319.TEZ.order_history`
"""
df = client.query(query).to_dataframe()

In [ ]:
import pandas as pd

# 🟠 1. Tarih sütununu düzgün biçime çevir
# Eğer tarih formatı "13.04.2025" gibi ise:
df['purchase_date'] = pd.to_datetime(df['purchase_date'], format='%d.%m.%Y', errors='coerce')

# 🟠 2. Harcama sütununu (Monetary) sayıya çevir
# Eğer içinde ₺ işareti veya virgül varsa, bunları temizle
if df['total_price'].dtype == 'object':
    df['total_price'] = df['total_price'].str.replace('₺','', regex=False)
    df['total_price'] = df['total_price'].str.replace(',','.', regex=False)

# Sayıya çevir, hatalı değerleri NaN yap
df['total_price'] = pd.to_numeric(df['total_price'], errors='coerce')

# 🟠 3. Referans tarihi belirle (en son satın alma tarihinden bir gün sonrası)
ref_date = df['purchase_date'].max() + pd.Timedelta(days=1)

# 🟠 4. RFM hesaplama
rfm = df.groupby('customer_id').agg({
    'purchase_date': lambda x: (ref_date - x.max()).days,    # Recency
    'customer_id': 'count',                                   # Frequency
    'total_price': 'sum'                                      # Monetary
}).rename(columns={
    'purchase_date': 'Recency',
    'customer_id': 'Frequency',
    'total_price': 'Monetary'
})

# 🟠 5. Veriyi temizle (boş değer varsa at)
rfm = rfm.dropna()

# 🟠 6. RFM skorları (1–5 arası)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)

# 🟠 7. RFM Skorunu birleştir
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
customer_id,,,,,,,
10019720,115,3,50148,1,5,2,152
10021856,52,5,62481,4,5,3,453
10024050,86,2,13636,2,3,1,231
10040204,94,1,8443,2,1,1,211
10054039,49,2,100747,4,3,4,434


In [ ]:
def segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4:
        return 'VIP Customers'
    elif row['R_Score'] >= 4:
        return 'Loyal Customers'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk Customers'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost Customers'
    else:
        return 'Other'

rfm['Segment'] = rfm.apply(segment, axis=1)

In [ ]:
rfm.reset_index(inplace=True)  # customer_id'yi tekrar kolona al

table_id = "lastproject-465319.TEZ.rfm_results"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # Var olan tabloyu siler ve yenisiyle değiştirir
)

job = client.load_table_from_dataframe(rfm, table_id, job_config=job_config)
job.result()  # İşin tamamlanmasını bekle

print("RFM tablosu başarıyla yüklendi.")

RFM tablosu başarıyla yüklendi.
